<a href="https://colab.research.google.com/github/notebooklm1958-tech/-Prediction-Project_Boston-House-Price/blob/main/canaux_de_prix.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Canaux de prix pour le trading avec Python
Analyse d'AAPL à partir d'un fichier CSV (colonnes : `Date, Close, High, Low, Open, Volume`).

**Canaux calculés :** Donchian, Bollinger, Keltner, régression linéaire.

Installation : `pip install pandas numpy matplotlib`

## 1. Imports et chargement du CSV

Ce bloc de code importe les bibliothèques nécessaires comme `pandas` pour la manipulation des données, `numpy` pour les opérations numériques et `matplotlib.pyplot` pour la visualisation. Il charge ensuite le fichier CSV spécifié par `CHEMIN_CSV` dans un DataFrame pandas, en convertissant la colonne 'Date' en format de date et en la définissant comme index. Enfin, il affiche le nombre de sessions et les 5 dernières lignes du DataFrame pour un aperçu.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CHEMIN_CSV = "AAPL.csv"   # placez le fichier dans le même dossier que le notebook

df = pd.read_csv(CHEMIN_CSV, parse_dates=["Date"], index_col="Date").sort_index()
df = df[["Open", "High", "Low", "Close", "Volume"]].dropna()
print(f"{len(df)} séances : {df.index[0].date()} -> {df.index[-1].date()}")
df.tail()

Ce bloc de code définit les fonctions pour calculer différents canaux de prix. Chacune de ces fonctions prend un DataFrame en entrée et renvoie un DataFrame contenant les bandes 'haut', 'milieu' et 'bas' du canal correspondant. `donchian` calcule le plus haut/bas sur N périodes, `bollinger` utilise la moyenne mobile et l'écart-type, `atr` calcule la moyenne réelle de l'amplitude (utilisée par Keltner), `keltner` utilise l'EMA et l'ATR, et `regression` calcule un canal basé sur une régression linéaire.

## 2. Les canaux de prix

| Canal | Principe |
|---|---|
| Donchian | Plus haut / plus bas sur N périodes |
| Bollinger | Moyenne mobile ± k écarts-types |
| Keltner | EMA ± k × ATR |
| Régression | Droite de tendance ± k écarts-types |

In [ ]:
def donchian(df, n=20):
    haut = df["High"].rolling(n).max()
    bas = df["Low"].rolling(n).min()
    return pd.DataFrame({"haut": haut, "milieu": (haut + bas) / 2, "bas": bas})


def bollinger(df, n=20, k=2.0):
    ma = df["Close"].rolling(n).mean()
    std = df["Close"].rolling(n).std()
    return pd.DataFrame({"haut": ma + k * std, "milieu": ma, "bas": ma - k * std})


def atr(df, n=14):
    pc = df["Close"].shift(1)
    tr = pd.concat([df["High"] - df["Low"],
                    (df["High"] - pc).abs(),
                    (df["Low"] - pc).abs()], axis=1).max(axis=1)
    return tr.ewm(alpha=1 / n, adjust=False).mean()


def keltner(df, n=20, k=2.0, n_atr=14):
    ema = df["Close"].ewm(span=n, adjust=False).mean()
    a = atr(df, n_atr)
    return pd.DataFrame({"haut": ema + k * a, "milieu": ema, "bas": ema - k * a})


def regression(df, n=100, k=2.0):
    y = df["Close"].iloc[-n:].values
    x = np.arange(n)
    pente, ordonnee = np.polyfit(x, y, 1)
    ligne = pente * x + ordonnee
    sigma = np.std(y - ligne)
    return pd.DataFrame({"haut": ligne + k * sigma, "milieu": ligne, "bas": ligne - k * sigma},
                        index=df.index[-n:])

Ce bloc de code calcule les différents canaux de prix pour le DataFrame `df` en utilisant les fonctions définies précédemment. Les résultats sont stockés dans le dictionnaire `canaux`, où chaque clé est le nom du canal et chaque valeur est le DataFrame des bandes du canal. Enfin, il affiche la valeur des bandes 'haut', 'milieu' et 'bas' pour la dernière période pour chaque canal.

## 3. Calcul des canaux (paramètres modifiables)

In [ ]:
canaux = {
    "Canal de Donchian (20)": donchian(df, 20),
    "Bandes de Bollinger (20, 2)": bollinger(df, 20, 2),
    "Canal de Keltner (20, 2 ATR)": keltner(df, 20, 2),
    "Canal de régression (100 derniers jours)": regression(df, 100, 2),
}
{nom: c.iloc[-1].round(2).to_dict() for nom, c in canaux.items()}

Ce bloc de code génère un graphique pour chaque canal de prix calculé. Il crée une figure avec plusieurs sous-graphiques (un par canal). Pour chaque sous-graphique, il trace le prix de clôture, ainsi que les bandes haute, médiane et basse du canal. La zone entre les bandes haute et basse est également remplie pour une meilleure visualisation. Enfin, il ajoute un titre, une légende et une grille à chaque graphique et les affiche.

## 4. Graphiques

In [ ]:
fig, axes = plt.subplots(len(canaux), 1, figsize=(13, 5 * len(canaux)), sharex=True)
for ax, (titre, c) in zip(axes, canaux.items()):
    ax.plot(df.index, df["Close"], color="black", lw=1, label="Clôture")
    ax.plot(c.index, c["haut"], "g--", label="Haute")
    ax.plot(c.index, c["milieu"], "b-", alpha=0.6, label="Milieu")
    ax.plot(c.index, c["bas"], "r--", label="Basse")
    ax.fill_between(c.index, c["bas"], c["haut"], alpha=0.08)
    ax.set_title(titre)
    ax.legend(loc="upper left")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Ce bloc de code définit une fonction `position_dans_canal` qui calcule la position relative du prix de clôture à l'intérieur d'un canal de prix donné (entre 0% et 100%). Il applique ensuite cette fonction pour calculer la position actuelle du prix de clôture pour chaque canal défini dans le dictionnaire `canaux`. Les résultats sont présentés dans un DataFrame, formatés en pourcentages.

## 5. Position du prix dans le canal
`0 %` = bande basse, `100 %` = bande haute.

In [ ]:
def position_dans_canal(close, canal):
    return (close - canal["bas"]) / (canal["haut"] - canal["bas"])

positions = {nom: position_dans_canal(df["Close"].loc[c.index], c).iloc[-1]
             for nom, c in canaux.items()}
pd.Series(positions).map("{:.0%}".format).to_frame("Position actuelle")

Ce bloc de code définit une fonction `signaux_donchian` qui génère des signaux d'achat (1) ou de vente (-1) basés sur la cassure du canal de Donchian. Un signal d'achat est généré si le prix de clôture actuel dépasse la bande haute de la veille, et un signal de vente si le prix de clôture actuel tombe en dessous de la bande basse de la veille. Cela permet d'éviter le 'look-ahead bias'. Le signal est ensuite ajouté en tant que nouvelle colonne 'signal' au DataFrame `df`, et les 10 dernières lignes où un signal a été généré sont affichées.

## 6. Signaux de cassure (Donchian)
On compare la clôture au canal de **la veille** (`shift(1)`) pour éviter le look-ahead bias.

In [ ]:
def signaux_donchian(df, n=20):
    c = donchian(df, n)
    s = pd.Series(0, index=df.index)
    s[df["Close"] > c["haut"].shift(1)] = 1     # cassure haussière -> achat
    s[df["Close"] < c["bas"].shift(1)] = -1     # cassure baissière -> vente
    return s

df["signal"] = signaux_donchian(df, 20)
df.loc[df["signal"] != 0, ["Close", "signal"]].tail(10)

---
*Ce notebook est un outil d'analyse technique et ne constitue pas un conseil en investissement. Testez toute stratégie sur un historique (frais et glissement inclus) avant tout usage réel.*